Plot D

In [ ]:
df_pd = genes_pleiotropy_categories.toPandas()

# Explode the 'source' column to get one row per gene-category pair
df_exploded = df_pd.explode("source")

# Calculate ORs for each category using Logistic Regression
# Model: In_Category ~ uniqueDiseases
results = []
all_categories = df_exploded["source"].unique()
# df_pd["uniqueDiseases"] = np.log(df_pd["uniqueDiseases"])
for category in all_categories:
    if category is None:
        continue

    # Identify genes in this category
    genes_in_cat_ids = df_exploded[df_exploded["source"] == category]["geneId"].unique()

    # Define Target Variable: 1 if gene is in category, 0 otherwise
    df_pd["in_category"] = df_pd["geneId"].isin(genes_in_cat_ids).astype(int)

    # Skip categories with too few or too many genes (perfect separation risk)
    n_in_cat = df_pd["in_category"].sum()
    # Define Predictor: uniqueDiseases
    X = sm.add_constant(df_pd["uniqueDiseases"].astype(float))
    y = df_pd["in_category"]

    try:
        # Fit Logistic Regression
        model = sm.Logit(y, X).fit(disp=0)

        # Extract coefficients for uniqueDiseases
        log_or = model.params["uniqueDiseases"]
        p_value = model.pvalues["uniqueDiseases"]
        conf = model.conf_int()

        # 95% CI for Log Odds Ratio
        log_ci_lower = conf.loc["uniqueDiseases", 0]
        log_ci_upper = conf.loc["uniqueDiseases", 1]

        # Odds Ratio and its CI
        odds_ratio = np.exp(log_or)
        ci_lower = np.exp(log_ci_lower)
        ci_upper = np.exp(log_ci_upper)

        # Calculate percentage using the pre-join totals
        # This gives % of the category that is present in the pleiotropy dataset
        total_in_category = category_totals.get(category, 0)
        pct_overlap = (n_in_cat / total_in_category * 100) if total_in_category > 0 else 0

        label = f"{category} ({total_in_category}/{pct_overlap:.1f}%)"

        results.append(
            {
                "category": category,
                "label": label,
                "odds_ratio": odds_ratio,
                "log_odds_ratio": log_or,
                "ci_lower": ci_lower,
                "ci_upper": ci_upper,
                "log_ci_lower": log_ci_lower,
                "log_ci_upper": log_ci_upper,
                "p_value": p_value,
                "count": n_in_cat,
            }
        )
    except Exception as e:
        print(f"Skipping {category} due to error: {e}")

results_df = pd.DataFrame(results).sort_values("log_odds_ratio", ascending=True)

# Plotting
plt.figure(figsize=(10, 8))

# Filter out categories with NaN CIs
plot_data = results_df.copy()

y_pos = np.arange(len(plot_data))

plt.errorbar(
    x=plot_data["log_odds_ratio"],
    y=y_pos,
    xerr=[
        plot_data["log_odds_ratio"] - plot_data["log_ci_lower"],
        plot_data["log_ci_upper"] - plot_data["log_odds_ratio"],
    ],
    fmt="o",
    color="steelblue",
    ecolor="steelblue",
    capsize=3,
)

# Apply specific colors to markers
for i, (idx, row) in enumerate(plot_data.iterrows()):
    color = "red" if "gwas" in row["category"].lower() else "steelblue"
    plt.plot(row["log_odds_ratio"], i, "o", color=color)

plt.yticks(y_pos, plot_data["label"].tolist())
plt.axvline(x=0, color="gray", linestyle="--", linewidth=0.8)
plt.xlabel("Log Odds Ratio (per increase in number of unique diseases)")
plt.title("Enrichment of Pleiotropy in Gene Sets")
plt.grid(axis="x", linestyle=":", alpha=0.6)

plt.tight_layout()
plt.show()
